# Segundo Checkpoint — Modern Data Architecture & Engineering
Giovanni Pascon Corrêa — RM571546 — entrega (*)  
Vitor Matias do Nascimento — RM572761

Schema: workspace.rm571546.

**Consolidação parcial das evidências:** os códigos das questões 1 a 3 foram reconstruídos a partir das células fornecidas durante a atividade. Suas saídas originais não estão neste export, exceto a leitura e a contagem da questão 2. As questões 4 a 8 preservam as saídas do notebook exportado pelo aluno. Este arquivo ainda não atende ao requisito de todas as células executadas: para isso, é necessário incorporar o export do primeiro notebook.

**Não use Run all no ambiente atual.** Julho já foi carregado e o RESTORE já foi realizado. Não repita o append nem o UPDATE. A falha de schema da questão 3 é esperada e deve ser executada isoladamente no fluxo inicial. A consulta da questão 6 utiliza a saída correta obtida após a restauração; a consulta intermediária com valores corrompidos foi excluída.


## Questão 1 — Schema, Volume e tamanhos
Código reconstruído; saída original pendente.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.rm571546;
CREATE VOLUME IF NOT EXISTS workspace.rm571546.atividade;
USE CATALOG workspace;
USE SCHEMA rm571546;

Enviar vendas.parquet, vendas_julho.parquet e vendas.csv pela interface Catalog → workspace → rm571546 → atividade → Upload to this volume.

In [ ]:
caminho = "/Volumes/workspace/rm571546/atividade/"
for arquivo in dbutils.fs.ls(caminho):
    print(f"{arquivo.name}: {arquivo.size / (1024 * 1024):.2f} MB")

## Questão 2 — Leitura do semestre e Bronze
A leitura abaixo preserva a saída real do export recebido. A criação da tabela foi reconstruída e precisa da saída do primeiro notebook.

In [0]:
df_semestre = spark.read.parquet(
    "/Volumes/workspace/rm571546/atividade/vendas.parquet"
)

df_semestre.printSchema()

linhas = df_semestre.count()
colunas = len(df_semestre.columns)

print(f"Quantidade de linhas: {linhas}")
print(f"Quantidade de colunas: {colunas}")

assert linhas == 50000
assert colunas == 40

root
 |-- id_venda: long (nullable = true)
 |-- data: timestamp_ntz (nullable = true)
 |-- sku: long (nullable = true)
 |-- produto_nome: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- id_loja: long (nullable = true)
 |-- loja_nome: string (nullable = true)
 |-- loja_cidade: string (nullable = true)
 |-- loja_uf: string (nullable = true)
 |-- loja_regiao: string (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- cliente_faixa_etaria: string (nullable = true)
 |-- cliente_cidade: string (nullable = true)
 |-- cliente_segmento: string (nullable = true)
 |-- quantidade: long (nullable = true)
 |-- valor_unitario: double (nullable = true)
 |-- desconto: double (nullable = true)
 |-- valor_venda: double (nullable = true)
 |-- atributo_01: string (nullable = true)
 |-- atributo_02: string (nullable = true)
 |-- atributo_03: string (nullable = true)
 |-- atributo_04: string (nullable = true)
 |-- atributo_05: string (nul

In [ ]:
df_semestre.createOrReplaceTempView("vendas_semestre_temp")
spark.sql("""
    CREATE TABLE workspace.rm571546.bronze_vendas
    USING DELTA
    AS SELECT * FROM vendas_semestre_temp
""")
print("Tabela bronze_vendas criada.")

In [ ]:
%sql
SELECT COUNT(*) AS total_linhas FROM workspace.rm571546.bronze_vendas;

In [ ]:
%sql
DESCRIBE HISTORY workspace.rm571546.bronze_vendas;

## Questão 3 — Schema enforcement, atomicidade e evolução
Códigos reconstruídos; saídas originais pendentes. Não executar esta seção na Bronze já carregada.

In [ ]:
caminho = "/Volumes/workspace/rm571546/atividade/"
df_julho = spark.read.parquet(caminho + "vendas_julho.parquet")
print(f"Linhas de julho: {df_julho.count()}")
print(f"Colunas de julho: {len(df_julho.columns)}")
df_julho.printSchema()

### 3a — Erro esperado de schema
Execute isoladamente no fluxo inicial. Registre a mensagem e continue para 3b; não habilite mergeSchema nesta tentativa.

In [ ]:
df_julho.write.format("delta").mode("append").saveAsTable("workspace.rm571546.bronze_vendas")

### 3b — Histórico e contagem após a falha

In [ ]:
%sql
DESCRIBE HISTORY workspace.rm571546.bronze_vendas;

In [ ]:
%sql
SELECT COUNT(*) AS total_linhas FROM workspace.rm571546.bronze_vendas;

### 3c — Append com evolução: somente uma vez

In [ ]:
total_antes = spark.table("workspace.rm571546.bronze_vendas").count()
assert total_antes == 50000, f"Bronze com {total_antes} linhas. Carga interrompida para evitar duplicar julho."
df_julho.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("workspace.rm571546.bronze_vendas")
print("Julho acrescentado com evolução de schema.")

In [ ]:
%sql
SELECT canal_venda, COUNT(*) AS quantidade_linhas
FROM workspace.rm571546.bronze_vendas
GROUP BY canal_venda ORDER BY canal_venda;

In [ ]:
%sql
SELECT COUNT(*) AS total_linhas FROM workspace.rm571546.bronze_vendas;

In [ ]:
%sql
DESCRIBE HISTORY workspace.rm571546.bronze_vendas;

## Questão 4 — Camada Prata
Saídas preservadas do notebook exportado.

In [0]:
from pyspark.sql import functions as F
from decimal import Decimal

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA rm571546")

df_bronze = spark.table("workspace.rm571546.bronze_vendas")
assert df_bronze.count() == 58850, "Bronze diferente de 58.850 linhas; pare e envie o resultado."
assert len(df_bronze.columns) == 41, "Bronze deve ter 41 colunas."
print("Contexto: workspace.rm571546; Bronze validada com 58.850 linhas e 41 colunas.")
     

Contexto: workspace.rm571546; Bronze validada com 58.850 linhas e 41 colunas.


In [0]:
colunas_prata = [
    "id_venda", "data", "sku", "produto_nome", "categoria", "marca",
    "id_loja", "loja_nome", "loja_cidade", "loja_uf", "loja_regiao",
    "id_cliente", "cliente_faixa_etaria", "cliente_cidade", "cliente_segmento",
    "quantidade", "valor_unitario", "desconto", "valor_venda", "canal_venda"
]

df_base = (
    df_bronze.select(*colunas_prata)
    .withColumn("data", F.to_date("data"))
    .withColumn("loja_uf", F.upper(F.trim(F.col("loja_uf"))))
    .withColumn("valor_venda", F.col("valor_venda").cast("decimal(12,2)"))
    .withColumn("canal_venda", F.coalesce(F.col("canal_venda"), F.lit("Loja Física")))
)

# A ordem exigida é deduplicar primeiro e filtrar depois.
df_sem_dup = df_base.dropDuplicates(["id_venda"])
df_prata = df_sem_dup.filter(F.col("valor_venda") > 0)

n_bronze = df_bronze.count()
n_sem_dup = df_sem_dup.count()
n_prata = df_prata.count()

resumo_qualidade = spark.createDataFrame([
    (1, "Linhas na bronze_vendas", n_bronze),
    (2, "Linhas após remover duplicidades", n_sem_dup),
    (3, "Linhas removidas por duplicidade", n_bronze - n_sem_dup),
    (4, "Linhas removidas por valor inválido", n_sem_dup - n_prata),
    (5, "Linhas na prata_vendas", n_prata),
], ["ordem", "etapa", "quantidade_linhas"])

display(resumo_qualidade.orderBy("ordem").drop("ordem"))
assert len(df_prata.columns) == 20
assert df_prata.filter(F.col("data").isNull()).count() == 0

df_prata.createOrReplaceTempView("prata_preparada_temp")
spark.sql("""
    CREATE OR REPLACE TABLE workspace.rm571546.prata_vendas
    USING DELTA AS SELECT * FROM prata_preparada_temp
""")
print(f"Prata gravada: {n_prata} linhas e {len(df_prata.columns)} colunas.")


etapa,quantidade_linhas
Linhas na bronze_vendas,58850
Linhas após remover duplicidades,58700
Linhas removidas por duplicidade,150
Linhas removidas por valor inválido,100
Linhas na prata_vendas,58600


Prata gravada: 58600 linhas e 20 colunas.


In [0]:

%sql
SELECT DISTINCT loja_uf
FROM workspace.rm571546.prata_vendas
ORDER BY 1;
     

loja_uf
MG
PE
PR
RJ
RS
SC
SP


In [0]:

df_prata_salva = spark.table("workspace.rm571546.prata_vendas")
df_prata_salva.printSchema()
assert df_prata_salva.select("loja_uf").distinct().count() == 7
assert df_prata_salva.select("id_venda").distinct().count() == df_prata_salva.count()
print("Validação concluída: 20 colunas, 7 UFs e id_venda sem duplicidade.")
     

root
 |-- id_venda: long (nullable = true)
 |-- data: date (nullable = true)
 |-- sku: long (nullable = true)
 |-- produto_nome: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- id_loja: long (nullable = true)
 |-- loja_nome: string (nullable = true)
 |-- loja_cidade: string (nullable = true)
 |-- loja_uf: string (nullable = true)
 |-- loja_regiao: string (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- cliente_faixa_etaria: string (nullable = true)
 |-- cliente_cidade: string (nullable = true)
 |-- cliente_segmento: string (nullable = true)
 |-- quantidade: long (nullable = true)
 |-- valor_unitario: double (nullable = true)
 |-- desconto: double (nullable = true)
 |-- valor_venda: decimal(12,2) (nullable = true)
 |-- canal_venda: string (nullable = true)

Validação concluída: 20 colunas, 7 UFs e id_venda sem duplicidade.


## Questão 5 — Star schema e integridade

In [0]:

df_p = spark.table("workspace.rm571546.prata_vendas")

tabelas_ouro = {
    "dim_produto": df_p.select("sku", "produto_nome", "categoria", "marca").distinct(),
    "dim_loja": df_p.select("id_loja", "loja_nome", "loja_cidade", "loja_uf", "loja_regiao").distinct(),
    "dim_cliente": df_p.select("id_cliente", "cliente_faixa_etaria", "cliente_cidade", "cliente_segmento").distinct(),
    "dim_tempo": df_p.select("data").distinct()
        .withColumn("ano", F.year("data"))
        .withColumn("trimestre", F.quarter("data"))
        .withColumn("mes", F.month("data")),
    "dim_canal": df_p.select("canal_venda").distinct(),
    "fato_vendas": df_p.select(
        "id_venda", "data", "sku", "id_loja", "id_cliente", "canal_venda",
        "quantidade", "valor_unitario", "desconto", "valor_venda"
    )
}

for nome, dataframe in tabelas_ouro.items():
    dataframe.createOrReplaceTempView("ouro_preparado_temp")
    spark.sql(f"CREATE OR REPLACE TABLE workspace.rm571546.{nome} USING DELTA AS SELECT * FROM ouro_preparado_temp")
    print(f"Tabela criada: {nome}")
     

Tabela criada: dim_produto
Tabela criada: dim_loja
Tabela criada: dim_cliente
Tabela criada: dim_tempo
Tabela criada: dim_canal
Tabela criada: fato_vendas


In [0]:
%sql
SELECT 'dim_produto' AS tabela, COUNT(*) AS quantidade_linhas FROM workspace.rm571546.dim_produto
UNION ALL
SELECT 'dim_loja', COUNT(*) FROM workspace.rm571546.dim_loja
UNION ALL
SELECT 'dim_cliente', COUNT(*) FROM workspace.rm571546.dim_cliente
UNION ALL
SELECT 'dim_tempo', COUNT(*) FROM workspace.rm571546.dim_tempo
UNION ALL
SELECT 'dim_canal', COUNT(*) FROM workspace.rm571546.dim_canal;

tabela,quantidade_linhas
dim_produto,800
dim_loja,40
dim_cliente,5000
dim_tempo,211
dim_canal,3


In [0]:

%sql
SELECT id_loja, COUNT(*) AS quantidade
FROM workspace.rm571546.dim_loja
GROUP BY id_loja
HAVING COUNT(*) > 1;

id_loja,quantidade


In [0]:
soma_prata = spark.table("workspace.rm571546.prata_vendas").agg(F.sum("valor_venda").alias("total")).first()["total"]
soma_fato = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda").alias("total")).first()["total"]

display(spark.createDataFrame([
    ("prata_vendas", soma_prata),
    ("fato_vendas", soma_fato)
], "tabela string, faturamento decimal(20,2)"))

assert soma_prata == soma_fato, "Faturamentos diferentes: pare antes de continuar."
assert spark.table("workspace.rm571546.dim_loja").count() == 40
total_referencia = soma_fato
print(f"Integridade confirmada. Faturamento de referência para Q8: {total_referencia}")

tabela,faturamento
prata_vendas,32073054.62
fato_vendas,32073054.62


Integridade confirmada. Faturamento de referência para Q8: 32073054.62


In [0]:

%sql
SELECT COUNT(*) AS lojas_sem_tratamento
FROM (
    SELECT DISTINCT id_loja, loja_nome, loja_cidade, loja_uf, loja_regiao
    FROM workspace.rm571546.bronze_vendas
) lojas;

lojas_sem_tratamento
208


## Questão 6 — Evolução mensal e cobertura de datas
A saída abaixo é a consulta executada após RESTORE, com o faturamento correto. Foi movida para a seção correspondente sem modificar seus resultados.

In [0]:
%sql
WITH mensal AS (
    SELECT l.loja_regiao, t.ano, t.mes,
           SUM(f.valor_venda) AS faturamento
    FROM workspace.rm571546.fato_vendas f
    JOIN workspace.rm571546.dim_loja l ON f.id_loja = l.id_loja
    JOIN workspace.rm571546.dim_tempo t ON f.data = t.data
    WHERE t.ano = 2026 AND t.mes BETWEEN 1 AND 7
    GROUP BY l.loja_regiao, t.ano, t.mes
), comparacao AS (
    SELECT *,
           LAG(faturamento) OVER (
               PARTITION BY loja_regiao ORDER BY ano, mes
           ) AS faturamento_anterior
    FROM mensal
)
SELECT loja_regiao, mes, faturamento, faturamento_anterior,
       ROUND((faturamento - faturamento_anterior)
             / NULLIF(faturamento_anterior, 0) * 100, 2) AS variacao_pct
FROM comparacao
ORDER BY loja_regiao, ano, mes;

loja_regiao,mes,faturamento,faturamento_anterior,variacao_pct
Nordeste,1,226358.95,null,null
Nordeste,2,212022.85,226358.95,-6.33
Nordeste,3,235427.32,212022.85,11.04
Nordeste,4,227040.98,235427.32,-3.56
Nordeste,5,236771.77,227040.98,4.29
Nordeste,6,230442.52,236771.77,-2.67
Nordeste,7,425996.97,230442.52,84.86
Sudeste,1,3055724.04,null,null
Sudeste,2,2809384.61,3055724.04,-8.06
Sudeste,3,3103331.01,2809384.61,10.46


In [0]:

%sql
SELECT ano, mes, COUNT(DISTINCT data) AS dias_presentes,
       DAY(LAST_DAY(MIN(data))) AS dias_calendario,
       MIN(data) AS primeira_data, MAX(data) AS ultima_data
FROM workspace.rm571546.dim_tempo
WHERE ano = 2026
GROUP BY ano, mes
ORDER BY ano, mes;

ano,mes,dias_presentes,dias_calendario,primeira_data,ultima_data
2026,1,31,31,2026-01-01,2026-01-31
2026,2,28,28,2026-02-01,2026-02-28
2026,3,31,31,2026-03-01,2026-03-31
2026,4,30,30,2026-04-01,2026-04-30
2026,5,31,31,2026-05-01,2026-05-31
2026,6,29,30,2026-06-01,2026-06-29
2026,7,31,31,2026-07-01,2026-07-31


## Questão 7 — Canais em julho

In [0]:
%sql
SELECT c.canal_venda,
       SUM(f.valor_venda) AS faturamento,
       COUNT(DISTINCT f.id_venda) AS vendas,
       ROUND(SUM(f.valor_venda) / COUNT(DISTINCT f.id_venda), 2) AS ticket_medio,
       ROUND(100 * SUM(f.valor_venda)
             / SUM(SUM(f.valor_venda)) OVER (), 2) AS participacao_pct,
       ROUND(100.0 * COUNT(DISTINCT f.id_venda)
             / SUM(COUNT(DISTINCT f.id_venda)) OVER (), 2) AS participacao_vendas_pct
FROM workspace.rm571546.fato_vendas f
JOIN workspace.rm571546.dim_canal c ON f.canal_venda = c.canal_venda
JOIN workspace.rm571546.dim_tempo t ON f.data = t.data
WHERE t.ano = 2026 AND t.mes = 7
GROUP BY c.canal_venda
ORDER BY faturamento DESC;
     

canal_venda,faturamento,vendas,ticket_medio,participacao_pct,participacao_vendas_pct
Loja Física,3227464.59,5923,544.90,67.42,68.87
Aplicativo,896539.02,1723,520.34,18.73,20.03
Site,663050.81,954,695.02,13.85,11.09


## Questão 8 — UPDATE, Time Travel e RESTORE
Saídas originais preservadas. Não executar o UPDATE outra vez.

In [0]:
historico_antes = spark.sql("DESCRIBE HISTORY workspace.rm571546.fato_vendas")
assert historico_antes.filter(F.col("operation") == "UPDATE").count() == 0, (
    "Já existe UPDATE no histórico. Não repita o erro; envie o histórico para análise."
)
versao_antes_update = historico_antes.agg(F.max("version")).first()[0]
total_referencia = spark.table("workspace.rm571546.prata_vendas").agg(F.sum("valor_venda")).first()[0]
total_antes_update = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda")).first()[0]
total_julho_antes = spark.sql("""
    SELECT SUM(valor_venda) FROM workspace.rm571546.fato_vendas
    WHERE data >= '2026-07-01'
""").first()[0]
assert total_antes_update == total_referencia
print(f"Versão antes do UPDATE: {versao_antes_update}")
print(f"Faturamento correto: {total_referencia}")
print(f"Faturamento de julho antes do erro: {total_julho_antes}")
     

Versão antes do UPDATE: 0
Faturamento correto: 32073054.62
Faturamento de julho antes do erro: 4787054.42


In [0]:

%sql
UPDATE fato_vendas SET valor_venda = valor_venda * 100 WHERE data >= '2026-07-01';

num_affected_rows
8600


In [0]:

%sql
SELECT SUM(valor_venda) AS faturamento_apos_erro
FROM workspace.rm571546.fato_vendas;

faturamento_apos_erro
505991442.20


In [0]:

%sql
DESCRIBE HISTORY workspace.rm571546.fato_vendas;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-10-07T12:28:16.000Z,73962701829020,giovanni.pscorrea@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2991471707770785),41e65b29-f86b-4000-9d16-b97ee6da701b,1007-120127-o5g56au9-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 1032184, p25FileSize -> 785509, numDeletionVectorsRemoved -> 1, minFileSize -> 785509, numAddedFiles -> 1, maxFileSize -> 785509, p75FileSize -> 785509, p50FileSize -> 785509, numAddedBytes -> 785509)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-07T12:28:14.000Z,73962701829020,giovanni.pscorrea@gmail.com,UPDATE,"Map(predicate -> [""(data#22978 >= 2026-07-01)""])",null,List(2991471707770785),41e65b29-f86b-4000-9d16-b97ee6da701b,1007-120127-o5g56au9-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4102, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1892, numAddedFiles -> 1, numUpdatedRows -> 8600, numAddedBytes -> 152471, rewriteTimeMs -> 2177)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-07T12:24:18.000Z,73962701829020,giovanni.pscorrea@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""aba1c151-1709-4df3-8672-9b42bbfce4d3""}, statsOnLoad -> true)",null,List(2991471707770785),16ee9b7c-7730-4b5c-83e0-8f6b85bff0b0,1007-120127-o5g56au9-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 58600, numOutputBytes -> 879713)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
historico = spark.sql("DESCRIBE HISTORY workspace.rm571546.fato_vendas")
versao_update = historico.filter(F.col("operation") == "UPDATE").agg(F.max("version")).first()[0]
assert versao_update is not None, "Nenhum UPDATE encontrado no histórico."
versao_correta = historico.filter(F.col("version") < versao_update).agg(F.max("version")).first()[0]
assert versao_correta is not None
assert versao_correta == versao_antes_update, "Histórico mudou inesperadamente; envie as capturas."

total_apos_erro = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda")).first()[0]
assert total_apos_erro == total_referencia + Decimal(99) * total_julho_antes

consulta_time_travel = f"""
    SELECT SUM(valor_venda) AS faturamento_na_versao_correta
    FROM workspace.rm571546.fato_vendas VERSION AS OF {versao_correta}
"""
print(f"UPDATE na versão {versao_update}; versão correta anterior: {versao_correta}")
print(consulta_time_travel)
resultado_time_travel = spark.sql(consulta_time_travel)
display(resultado_time_travel)
assert resultado_time_travel.first()[0] == total_referencia

UPDATE na versão 1; versão correta anterior: 0

    SELECT SUM(valor_venda) AS faturamento_na_versao_correta
    FROM workspace.rm571546.fato_vendas VERSION AS OF 0



faturamento_na_versao_correta
32073054.62


In [0]:

comando_restore = f"RESTORE TABLE workspace.rm571546.fato_vendas TO VERSION AS OF {versao_correta}"
print(comando_restore)
display(spark.sql(comando_restore))

RESTORE TABLE workspace.rm571546.fato_vendas TO VERSION AS OF 0


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
879713,1,1,1,785509,879713


In [0]:

total_restaurado = spark.table("workspace.rm571546.fato_vendas").agg(F.sum("valor_venda")).first()[0]
display(spark.createDataFrame([
    ("Referência da questão 5b", total_referencia),
    ("Fato após RESTORE", total_restaurado)
], "etapa string, faturamento decimal(20,2)"))
assert total_restaurado == total_referencia, "O faturamento ainda não foi recuperado."
print("RESTORE confirmado: faturamento igual ao da questão 5b.")

etapa,faturamento
Referência da questão 5b,32073054.62
Fato após RESTORE,32073054.62


RESTORE confirmado: faturamento igual ao da questão 5b.


In [0]:

%sql
DESCRIBE HISTORY workspace.rm571546.fato_vendas;
     

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-10-07T12:31:31.000Z,73962701829020,giovanni.pscorrea@gmail.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(2991471707770785),1c1a2bf7-b43e-44ba-bf5c-7cbd41ee3842,1007-120127-o5g56au9-v2n,2,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 785509, numRemovedFiles -> 1, restoredFilesSize -> 879713, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 879713)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-07T12:28:16.000Z,73962701829020,giovanni.pscorrea@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2991471707770785),41e65b29-f86b-4000-9d16-b97ee6da701b,1007-120127-o5g56au9-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 1032184, p25FileSize -> 785509, numDeletionVectorsRemoved -> 1, minFileSize -> 785509, numAddedFiles -> 1, maxFileSize -> 785509, p75FileSize -> 785509, p50FileSize -> 785509, numAddedBytes -> 785509)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-07T12:28:14.000Z,73962701829020,giovanni.pscorrea@gmail.com,UPDATE,"Map(predicate -> [""(data#22978 >= 2026-07-01)""])",null,List(2991471707770785),41e65b29-f86b-4000-9d16-b97ee6da701b,1007-120127-o5g56au9-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4102, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1892, numAddedFiles -> 1, numUpdatedRows -> 8600, numAddedBytes -> 152471, rewriteTimeMs -> 2177)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-07T12:24:18.000Z,73962701829020,giovanni.pscorrea@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""aba1c151-1709-4df3-8672-9b42bbfce4d3""}, statsOnLoad -> true)",null,List(2991471707770785),16ee9b7c-7730-4b5c-83e0-8f6b85bff0b0,1007-120127-o5g56au9-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 58600, numOutputBytes -> 879713)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
